# Part 12 · Notebook 10 — Incidents, the go-live review and the track record

**Sessions:** S19–S20 (Incident response drills · Capstone kickoff & go-live review) and weeks 6–8 (S21–S24: weekly reviews, final review, defense) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Run an incident through its lifecycle and check the response-time SLAs.
2. Build a post-mortem timeline from the audit trail.
3. Hold the go-live review on evidence.
4. Review a 4-week live record every week against each strategy's backtest band: scale, hold or demote, and ramp capital.
5. Check the graduation requirements from the audit chain.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. The incident lifecycle

Detect → contain (pause or kill) → diagnose → recover → reconcile → close, in that order and forward in time. Each severity has response targets (`p.SLA_MINUTES`): a sev1 must be contained within 5 minutes and reconciled within 60. Allow only the **next** phase; refuse anything else.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyIncident(p.Incident):
    def advance(self, phase, at):
        i = p.PHASES.index(self.phase)
        if ...:                                                      # ✍️ there is no next phase, or `phase` is not it
            raise ValueError(f"{self.phase} cannot go to {phase}")
        at = pd.Timestamp(at)
        if at < self.times[self.phase]:
            raise ValueError("time goes forward")
        self.times[phase] = at

def drill(cls):
    inc = cls("runaway order loop in mom", "sev1", "2025-11-12 10:02")
    log = []
    for phase, at in (("contained", "10:05"), ("recovered", "10:20"), ("diagnosed", "10:31"), ("recovered", "10:48"),
                      ("reconciled", "11:15"), ("closed", "09:00"), ("closed", "16:30")):
        try:
            inc.advance(phase, f"2025-11-12 {at}")
            log.append(f"{at} {phase}")
        except ValueError as err:
            log.append(f"{at} refused: {err}")
    return log, inc.sla_breaches()

log, breaches = p.check("Incident.advance", p.attempt(drill, MyIncident), drill(p.Incident))
print("\n".join(log))
print("SLA breaches:", breaches)

Recovering before diagnosing is refused, and so is a time earlier than the previous phase (09:00 typed for 16:30). The team contained the loop in 3 minutes, but reconciling took 73 minutes against a 60-minute target: that is an action item for the post-mortem.

## 2. The post-mortem timeline

A **blameless** post-mortem starts from facts, and the audit chain has them. `p.postmortem_timeline` cuts the records of the incident window, in time order:

In [ ]:
clock = iter(pd.Timestamp("2025-11-12 09:58") + pd.to_timedelta([0, 220, 261, 262, 263, 540, 1380, 2760, 4380, 9000], unit="s"))
audit = p.AuditLog(now=lambda: next(clock))
for event, data in [("order", {"id": "O900"}), ("alert", {"anomaly": "runaway_order_rate", "rate": 60}),
                    ("controller_action", {"action": "trip_kill_switch"}), ("killswitch_trip", {"by": "controller"}),
                    ("orders_cancelled", {"n": 57}), ("diagnosis", {"cause": "retry loop on a rejected order"}),
                    ("fix_deployed", {"version": "1.4.1"}), ("reconciled", {"breaks": 0}),
                    ("approval_granted", {"kind": "resume", "approver": "bob"}), ("daily_report", {})]:
    audit.append(event, **data)
timeline = p.postmortem_timeline(audit.records, "2025-11-12 10:00", "2025-11-12 11:30")
print("audit chain intact:", audit.verify() == -1)
pd.DataFrame(timeline, columns=["time", "event"])

The report around this timeline: summary, impact (orders, P&L, time without trading), root cause, what went well, what didn't, and action items with owners, such as a retry limit, and a check that a rejected order is never resent unchanged. Blameless means it asks *how the system let this happen*, not *who did it*.

## 3. The go-live review

Approval for small live is based on evidence only: every strategy passed the Part 8 validation gate (DSR ≥ 0.95, PBO < 0.5), at least 10 runbooks, every drill passed, the security checklist complete, a written rollout plan:

In [ ]:
evidence = {"strategies": {"momentum": {"dsr": 0.97, "pbo": 0.2}, "pairs": {"dsr": 0.96, "pbo": 0.3}, "options": {"dsr": 0.95, "pbo": 0.4}},
            "runbooks": 10, "drills": {"disconnect": True, "runaway": True, "bad_tick": True},
            "security_complete": True, "rollout_plan": True}
print("the evidence pack:", p.go_live_review(evidence))
weak = {**evidence, "strategies": {**evidence["strategies"], "options": {"dsr": 0.91, "pbo": 0.55}}, "runbooks": 7,
        "drills": {**evidence["drills"], "db_outage": False}}
print("a weaker pack:     ", p.go_live_review(weak))

## 4. Four weeks live: the weekly reviews

The track record starts. `p.track_record()` has three strategies; each week, the review compares each strategy's cumulative live return with its backtest band. Only falling **below** the band counts against a strategy (`lower_only=True`): beating it calls for an investigation, not a demotion.

In [ ]:
bt, live = p.track_record()
inside = pd.DataFrame({s: p.inside_band(live[s], p.backtest_band(bt[s], horizon=len(live)), lower_only=True).to_numpy()
                       for s in live.columns}, index=live.index)
print("share of live days inside the band:", inside.mean().round(2).to_dict())

Each weekly review decides per strategy, from that week's 5 days and incidents: **demote** if inside on fewer than half of the days or 2+ incidents; **scale** if inside every day with no incident; otherwise **hold**.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def weekly_decisions(inside, incidents):
    out = {}
    for s in inside.columns:
        share, inc = inside[s].mean(), incidents.get(s, 0)
        out[s] = ...                                                 # ✍️ "demote", "scale" or "hold"
    return out

incidents = {3: {"pairs": 1}}                                        # week 3: one incident on pairs (a partial fill on one leg)
weeks = {w: (inside.iloc[(w - 1) * 5: w * 5], incidents.get(w, {})) for w in range(1, 5)}
mine = p.check("weekly_decisions", {w: p.attempt(weekly_decisions, *a) for w, a in weeks.items()},
               {w: p.weekly_decisions(*a) for w, a in weeks.items()})
decisions = pd.DataFrame(mine).T.rename_axis("week")
decisions

The capital follows the decisions and the S17 ramp: a demotion sends a strategy back to paper (capital 0, and it stays there until it re-qualifies); otherwise `p.capital_ramp` with the number of consecutive "scale" weeks and the drawdown so far:

In [ ]:
PLAN = {"min": 0.1, "max": 1.0, "step": 0.1, "weeks_per_step": 2, "max_dd": -0.03}
capital = {}
for s in decisions.columns:
    frac, streak, path = 0.1, 0, []
    for w, dec in decisions[s].items():
        cum = pd.concat([pd.Series([0.0]), live[s].iloc[: w * 5].cumsum()], ignore_index=True)
        drawdown = float((cum - cum.cummax()).iloc[-1])
        if dec == "demote" or frac == 0:
            frac = 0.0
        else:
            streak = streak + 1 if dec == "scale" else 0
            frac = p.capital_ramp(frac, streak, drawdown, PLAN)
        path.append(frac)
    capital[s] = path
pd.DataFrame(capital, index=decisions.index).rename_axis("capital fraction after the review of week")

The options strategy is demoted at the week-2 review, while it still runs at the 10% starting size, so the lesson is cheap. Momentum and pairs earn their steps up slowly; one day below the band (momentum, week 2) or one incident (pairs, week 3) is enough to hold and restart the count. Every decision, with its reasons, goes into the audit log.

## 5. Graduation

Four requirements are checked from the evidence, not from slides: the kill switch was demonstrated, **no order bypassed the risk engine**, every end-of-day reconciliation break is explained, and every strategy has a DSR. The second comes from the audit chain: list the ids of `order` records that were not preceded by an approved `risk_decision` for that order.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def orders_without_risk_approval(records):
    approved, bad = set(), []
    for r in records:
        if r["event"] == "risk_decision" and r["data"].get("approved"):
            approved.add(r["data"]["order_id"])
        elif ...:                                                    # ✍️ an order whose id was not approved before it
            bad.append(r["data"]["id"])
    return bad

def audit_trail(n_orders=30, skip=(), late=(), rejected=()):
    log = p.AuditLog()
    for i in range(n_orders):
        oid = f"O{i:03d}"
        if oid in late:                                              # approved only AFTER it was sent
            log.append("order", id=oid)
            log.append("risk_decision", order_id=oid, approved=True)
            continue
        if oid not in skip:
            log.append("risk_decision", order_id=oid, approved=oid not in rejected)
        log.append("order", id=oid)
    log.append("killswitch_trip", reason="defense demo")
    return log.records

trails = {"clean": audit_trail(), "one order skipped the risk engine": audit_trail(skip={"O017"}),
          "approved after sending": audit_trail(late={"O023"}), "rejected but sent anyway": audit_trail(rejected={"O008"})}
p.check("orders_without_risk_approval", {k: p.attempt(orders_without_risk_approval, r) for k, r in trails.items()},
        {k: p.orders_without_risk_approval(r) for k, r in trails.items()})

In [ ]:
dsr = {"momentum": 0.97, "pairs": 0.96, "options": 0.95}
print("the capstone:", p.graduation(trails["clean"], {"2025-11-07": 0, "2025-11-14": 1}, {"2025-11-14"}, dsr))
print("a weaker one:", p.graduation(trails["one order skipped the risk engine"][:-1], {"2025-11-07": 0, "2025-11-14": 1, "2025-11-21": 2},
                                    {"2025-11-14"}, {**dsr, "options": None}))

## Wrap-up: the capstone

* Incidents follow a lifecycle with SLAs; post-mortems are blameless and built from the audit trail.
* Go-live, weekly decisions, capital and graduation are all decided on evidence, with rules written in advance.
* **S21–S22 weekly reviews:** performance against the band, attribution, reconciliation breaks, TCA, alerts and incidents, journal findings, and the decisions with their reasons.
* **S23–S24:** the 4-week results, lessons learned, and the defense: a 30-minute presentation, a live demo (including the kill switch) and questions from the panel.
* Graded versions: `labs/part12/week45_golive` (`Incident`, `postmortem_timeline`, `go_live_review`, `weekly_decisions`, `orders_without_risk_approval`, `graduation`) and the dossier in `labs/part12/clinic_w5_track_record`.